# 🔴 Notebook 02 – Hotspot Analysis
Dive into the DBSCAN-detected accident hotspots, their severity profiles, and geographic radius distributions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from pathlib import Path

sns.set_theme(style='whitegrid')
%matplotlib inline

DATA_DIR   = Path('data/processed')
df         = pd.read_csv(DATA_DIR / 'accidents_clean.csv')
hotspots   = pd.read_csv(DATA_DIR / 'hotspots.csv')
print(f'Accidents : {len(df):,}')
print(f'Hotspots  : {len(hotspots):,}')
hotspots.head(10)

## 1. Hotspot Summary Table
Top hotspots ranked by accident count.

In [ ]:
hotspots.sort_values('accident_count', ascending=False).head(15).style\
    .background_gradient(subset=['accident_count'], cmap='Reds')\
    .format({'severity_index': '{:.2f}', 'radius_m': '{:.0f}'})

## 2. Hotspots Plotted on Map
Circle size reflects accident count; colour encodes risk level.

In [ ]:
RISK_COLOR = {'HIGH': 'red', 'MODERATE': 'orange', 'LOW': 'green'}

m = folium.Map(location=[20.5, 78.9], zoom_start=5, tiles='CartoDB positron')
for _, row in hotspots.iterrows():
    color = RISK_COLOR.get(row['risk_level'], 'blue')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(5, min(25, row['accident_count'] / 5)),
        color=color, fill=True, fill_color=color, fill_opacity=0.6,
        popup=f"{row['hotspot_id']} | {row['risk_level']} | {row['accident_count']} accidents",
        tooltip=row['hotspot_id'],
    ).add_to(m)
m

## 3. Severity Index by Hotspot
Higher severity index → more severe/fatal accidents at that cluster.

In [ ]:
top_hs = hotspots.sort_values('accident_count', ascending=False).head(20)
fig, ax = plt.subplots(figsize=(14, 5))
colors = top_hs['risk_level'].map({'HIGH': '#e74c3c', 'MODERATE': '#f39c12', 'LOW': '#2ecc71'})
ax.bar(top_hs['hotspot_id'], top_hs['severity_index'], color=colors, edgecolor='white')
ax.set_xlabel('Hotspot ID')
ax.set_ylabel('Severity Index')
ax.set_title('Severity Index by Hotspot (top 20 by accident count)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Hotspot Radius Distribution
The spread of each cluster in metres.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(hotspots['radius_m'], bins=20, color='steelblue', edgecolor='white')
ax.set_xlabel('Cluster Radius (m)')
ax.set_ylabel('Number of Hotspots')
ax.set_title('Distribution of Hotspot Radii')
ax.axvline(hotspots['radius_m'].mean(), color='red', linestyle='--', label='Mean')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Risk Level Breakdown

In [ ]:
risk_summary = hotspots.groupby('risk_level').agg(
    hotspot_count=('hotspot_id', 'count'),
    total_accidents=('accident_count', 'sum'),
    avg_severity=('severity_index', 'mean'),
).reset_index()
print(risk_summary.to_string(index=False))

---
*End of Hotspot Analysis notebook.*